In [ ]:
import pandas as pd

## Look at no signal findings

In [ ]:
data = pd.read_csv('outputs/classified_samples.csv')

In [ ]:
no_signal = data[data['med_label'] == "NO_SIGNAL"]

In [ ]:
no_signal.groupby('bioproject')['bioproject'].count()

## Look at non mouse data

In [ ]:
all_data = pd.read_csv('data/combined_metadata_noncancer_removed.csv')

In [ ]:
all_data = all_data.iloc[:,:100]

In [ ]:
exclude = [
    "Mus musculus",
    "Mus musculus domesticus",
    "Mus musculus musculus"
]

non_mouse = all_data[~all_data['organism_scientific_name'].isin(exclude)]

In [ ]:
sample_400 = non_mouse.sample(n=400, random_state=42)

In [ ]:
sample_400

In [ ]:
cols = [
    "title",
    "single-cell/bulk",
    "tissue",
    "disease",
    "experiment_accession",
    "bioproject",
    "biosample",
    "sample_accession",
    "study_accession",
    "library_construction_protocol",
    "source_name",
    "run_accession",
    "cancer_type"
]

filtered_df = sample_400[cols]


In [ ]:
filtered_df

In [ ]:
filtered_df.to_csv("outputs/manual_label_not_mouse_ag.csv", index=False)


In [ ]:
# excel_data_full = pd.read_excel('outputs/manual_label_not_mouse.xlsx')

In [ ]:
# excel_data_raw = excel_data_full.drop("is_cancer")

## Run Classification Pipeline on Manual Label Data

In [1]:
import polars as pl
from functions import (
    classify_cancer_samples,
    initialize_medspacy_pipeline,
    generate_disease_rules,
    get_default_target_rules,
)
from text_column_processing import TextColumnConfig, preprocess_dataframe

In [2]:
# Convert pandas to polars for the pipeline
# excel_pl = pl.from_pandas(excel_data_raw)
# excel_data_full = pl.read_excel("outputs/manual_label_not_mouse.xlsx")


In [3]:
# excel_data_full = pl.read_excel("outputs/manual_label_not_mouse.xlsx")
# excel_data_raw = excel_data_full.drop("is_cancer")

# excel_pl = excel_data_raw

In [4]:
excel_pl = pl.read_excel("outputs/manual_label_not_mouse.xlsx")

Could not determine dtype for column 14, falling back to string


In [5]:
excel_pl=excel_pl.select(excel_pl.columns[:-1])

In [6]:
excel_pl.select(pl.all().exclude("cancer_type"))

title,single-cell/bulk,tissue,disease,experiment_accession,bioproject,biosample,sample_accession,study_accession,library_construction_protocol,source_name,run_accession,is_cancer
str,str,str,str,str,str,str,str,str,str,str,str,i64
"""GSM5860117: canine cortisol-se…","""bulk""","""nan nan nan nan nan nan""","""canine cortisol-secreting adre…","""SRX14042174""","""PRJNA803356""","""GSM5860117""","""SRS11877043""","""SRP358249""","""cells were trypsinized and sto…","""cortisol-secreting adrenocorti…","""SRR17882906""",1
"""GSM2453759: DLYM-1115_PENN03_v…","""bulk""",null,null,"""SRX2485257""","""PRJNA360981""","""GSM2453759""","""SRS1915489""","""SRP096582""","""Total RNA was extracted using …","""lymphoma""","""SRR5168428""",1
"""RNA-Seq of rabbit: male liver …","""bulk""","""rabbit liver cancer""",null,"""SRX26325965""","""PRJNA1168652""",null,"""SRS22852948""","""SRP537430""",null,null,"""SRR30923122""",1
"""RNA-Seq of melanoma : adult or…","""bulk""","""nan oral mucosal tumor nan nan…","""melanoma""","""SRX11560378""","""PRJNA749900""",null,"""SRS9598731""","""SRP329980""",null,null,"""SRR15255013""",1
"""GSM7678120: CCB010057; Canis l…","""bulk""","""nan osteosarcoma tumor tisue n…",null,"""SRX21232882""","""PRJNA1001809""","""GSM7678120""","""SRS18487976""","""SRP453113""","""Ambion miRvana kit. Illumina p…","""osteosarcoma tumor tisue""","""SRR25502047""",1
…,…,…,…,…,…,…,…,…,…,…,…,…
"""RNA-Seq of canine mammary tumo…","""bulk""","""nan tumor nan nan tumor nan""",null,"""SRX4634646""","""PRJNA489087""",null,"""SRS3735895""","""SRP159466""",null,null,"""SRR7779615""",null
"""RNA-Seq of melanoma : adult or…","""bulk""","""nan oral mucosal control nan n…","""non melanoma""","""SRX11560364""","""PRJNA749900""",null,"""SRS9598717""","""SRP329980""",null,null,"""SRR15255027""",null
"""RNA-Seq of melanoma : adult or…","""bulk""","""nan oral mucosal tumor nan nan…","""melanoma""","""SRX11560362""","""PRJNA749900""",null,"""SRS9598715""","""SRP329980""",null,null,"""SRR15255029""",null


In [7]:
# Remove cancer_type to prevent it from influencing classification
if "cancer_type" in excel_pl.columns:
    excel_pl = excel_pl.drop("cancer_type")
    print("✓ Removed cancer_type column from analysis")


✓ Removed cancer_type column from analysis


In [8]:
excel_pl

title,single-cell/bulk,tissue,disease,experiment_accession,bioproject,biosample,sample_accession,study_accession,library_construction_protocol,source_name,run_accession,is_cancer
str,str,str,str,str,str,str,str,str,str,str,str,i64
"""GSM5860117: canine cortisol-se…","""bulk""","""nan nan nan nan nan nan""","""canine cortisol-secreting adre…","""SRX14042174""","""PRJNA803356""","""GSM5860117""","""SRS11877043""","""SRP358249""","""cells were trypsinized and sto…","""cortisol-secreting adrenocorti…","""SRR17882906""",1
"""GSM2453759: DLYM-1115_PENN03_v…","""bulk""",null,null,"""SRX2485257""","""PRJNA360981""","""GSM2453759""","""SRS1915489""","""SRP096582""","""Total RNA was extracted using …","""lymphoma""","""SRR5168428""",1
"""RNA-Seq of rabbit: male liver …","""bulk""","""rabbit liver cancer""",null,"""SRX26325965""","""PRJNA1168652""",null,"""SRS22852948""","""SRP537430""",null,null,"""SRR30923122""",1
"""RNA-Seq of melanoma : adult or…","""bulk""","""nan oral mucosal tumor nan nan…","""melanoma""","""SRX11560378""","""PRJNA749900""",null,"""SRS9598731""","""SRP329980""",null,null,"""SRR15255013""",1
"""GSM7678120: CCB010057; Canis l…","""bulk""","""nan osteosarcoma tumor tisue n…",null,"""SRX21232882""","""PRJNA1001809""","""GSM7678120""","""SRS18487976""","""SRP453113""","""Ambion miRvana kit. Illumina p…","""osteosarcoma tumor tisue""","""SRR25502047""",1
…,…,…,…,…,…,…,…,…,…,…,…,…
"""RNA-Seq of canine mammary tumo…","""bulk""","""nan tumor nan nan tumor nan""",null,"""SRX4634646""","""PRJNA489087""",null,"""SRS3735895""","""SRP159466""",null,null,"""SRR7779615""",null
"""RNA-Seq of melanoma : adult or…","""bulk""","""nan oral mucosal control nan n…","""non melanoma""","""SRX11560364""","""PRJNA749900""",null,"""SRS9598717""","""SRP329980""",null,null,"""SRR15255027""",null
"""RNA-Seq of melanoma : adult or…","""bulk""","""nan oral mucosal tumor nan nan…","""melanoma""","""SRX11560362""","""PRJNA749900""",null,"""SRS9598715""","""SRP329980""",null,null,"""SRR15255029""",null


In [9]:
# Initialize MedSpaCy pipeline
cancer_rules, non_cancer_rules = get_default_target_rules()
nlp = initialize_medspacy_pipeline(cancer_rules, non_cancer_rules)

# Generate disease-specific rules from the data
unique_diseases = excel_pl.select("disease").drop_nulls().unique().to_series().to_list()
auto_rules, skipped = generate_disease_rules(unique_diseases, nlp, cancer_rules + non_cancer_rules)

if auto_rules:
    tm = nlp.get_pipe("medspacy_target_matcher")
    tm.add(auto_rules)
    print(f"Added {len(auto_rules)} auto-generated disease rules")


Added 18 auto-generated disease rules


In [10]:
# Preprocess text columns
config = TextColumnConfig()
excel_pl, col_tiers = preprocess_dataframe(excel_pl, config=config, include_discovered=False)
print(f"Created {len(col_tiers['normalized'])} normalized columns")

Created 3 normalized columns


In [11]:
excel_pl

title,single-cell/bulk,tissue,disease,experiment_accession,bioproject,biosample,sample_accession,study_accession,library_construction_protocol,source_name,run_accession,is_cancer,source_name_norm,tissue_norm,disease_norm
str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str
"""GSM5860117: canine cortisol-se…","""bulk""","""nan nan nan nan nan nan""","""canine cortisol-secreting adre…","""SRX14042174""","""PRJNA803356""","""GSM5860117""","""SRS11877043""","""SRP358249""","""cells were trypsinized and sto…","""cortisol-secreting adrenocorti…","""SRR17882906""",1,"""cortisol-secreting adrenocorti…","""nan nan nan nan nan nan""","""canine cortisol-secreting adre…"
"""GSM2453759: DLYM-1115_PENN03_v…","""bulk""",null,null,"""SRX2485257""","""PRJNA360981""","""GSM2453759""","""SRS1915489""","""SRP096582""","""Total RNA was extracted using …","""lymphoma""","""SRR5168428""",1,"""lymphoma""","""""",""""""
"""RNA-Seq of rabbit: male liver …","""bulk""","""rabbit liver cancer""",null,"""SRX26325965""","""PRJNA1168652""",null,"""SRS22852948""","""SRP537430""",null,null,"""SRR30923122""",1,"""""","""rabbit liver cancer""",""""""
"""RNA-Seq of melanoma : adult or…","""bulk""","""nan oral mucosal tumor nan nan…","""melanoma""","""SRX11560378""","""PRJNA749900""",null,"""SRS9598731""","""SRP329980""",null,null,"""SRR15255013""",1,"""""","""nan oral mucosal tumor nan nan…","""melanoma"""
"""GSM7678120: CCB010057; Canis l…","""bulk""","""nan osteosarcoma tumor tisue n…",null,"""SRX21232882""","""PRJNA1001809""","""GSM7678120""","""SRS18487976""","""SRP453113""","""Ambion miRvana kit. Illumina p…","""osteosarcoma tumor tisue""","""SRR25502047""",1,"""osteosarcoma tumor tisue""","""nan osteosarcoma tumor tisue n…",""""""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""RNA-Seq of canine mammary tumo…","""bulk""","""nan tumor nan nan tumor nan""",null,"""SRX4634646""","""PRJNA489087""",null,"""SRS3735895""","""SRP159466""",null,null,"""SRR7779615""",null,"""""","""nan tumor nan nan tumor nan""",""""""
"""RNA-Seq of melanoma : adult or…","""bulk""","""nan oral mucosal control nan n…","""non melanoma""","""SRX11560364""","""PRJNA749900""",null,"""SRS9598717""","""SRP329980""",null,null,"""SRR15255027""",null,"""""","""nan oral mucosal control nan n…","""non melanoma"""
"""RNA-Seq of melanoma : adult or…","""bulk""","""nan oral mucosal tumor nan nan…","""melanoma""","""SRX11560362""","""PRJNA749900""",null,"""SRS9598715""","""SRP329980""",null,null,"""SRR15255029""",null,"""""","""nan oral mucosal tumor nan nan…","""melanoma"""


In [12]:
# Run classification
predicted_df = classify_cancer_samples(excel_pl, nlp_pipeline=nlp, batch_size=64, use_normalized=True)


2026-03-03 10:16:00.072 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 0 'cortisol' marked as sentence start (span begin)
2026-03-03 10:16:00.072 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] Token/tag mapping: [(cortisol, True), (-, False), (secreting, False), (adrenocortical, False), (tumor, False)]
2026-03-03 10:16:00.074 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 0 'nan' marked as sentence start (span begin)
2026-03-03 10:16:00.074 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] Token/tag mapping: [(nan, True), (nan, False), (nan, False), (nan, False), (nan, False), (nan, False)]
2026-03-03 10:16:00.076 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 0 'canine' marked as sentence start (span begin)
2026-03-03 10:16:00.076 | DEBUG    | PyRuSH.PyRuSHSentenci

In [13]:
# Define label mapping (same as full_data.py)
FINAL_LABEL_MAP = {
    "confident_cancer": "CANCER",
    "likely_cancer": "CANCER",
    "confirmed_by_medspacy": "CANCER",
    "confirmed_non_cancer": "NON_CANCER",
    "likely_non_cancer": "NON_CANCER",
    "uncertain_no_signal": "NON_CANCER",
    "uncertain_weak_signal": "NON_CANCER",
    "uncertain_medspacy": "NON_CANCER",
    "NO_SIGNAL": "NON_CANCER"
}

# Use "med_label" instead of "confidence_category"
predicted_df = predicted_df.with_columns(
    pl.col("med_label").replace(FINAL_LABEL_MAP).alias("final_label")
)


In [14]:
# Re-add the is_cancer column from original excel_data
# is_cancer_series = pl.from_pandas(excel_data['is_cancer'])
is_cancer_series = excel_pl['is_cancer']
predicted_df = predicted_df.with_columns(is_cancer_series.alias("is_cancer"))


## Compare Predictions vs Ground Truth

In [15]:
# Show classification summary
print("=== Classification Summary ===")
summary = (
    predicted_df
    .group_by("final_label")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
print(summary)


=== Classification Summary ===
shape: (2, 2)
┌─────────────┬───────┐
│ final_label ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ CANCER      ┆ 207   │
│ NON_CANCER  ┆ 193   │
└─────────────┴───────┘


In [16]:
# Compare predictions to ground truth
comparison = (
    predicted_df
    .group_by("final_label", "is_cancer")
    .agg(pl.len().alias("count"))
    .sort("final_label", "is_cancer")
)
print("\n=== Predictions vs Ground Truth ===")
print(comparison)



=== Predictions vs Ground Truth ===
shape: (6, 3)
┌─────────────┬───────────┬───────┐
│ final_label ┆ is_cancer ┆ count │
│ ---         ┆ ---       ┆ ---   │
│ str         ┆ i64       ┆ u32   │
╞═════════════╪═══════════╪═══════╡
│ CANCER      ┆ null      ┆ 52    │
│ CANCER      ┆ 0         ┆ 2     │
│ CANCER      ┆ 1         ┆ 153   │
│ NON_CANCER  ┆ null      ┆ 48    │
│ NON_CANCER  ┆ 0         ┆ 66    │
│ NON_CANCER  ┆ 1         ┆ 79    │
└─────────────┴───────────┴───────┘


In [17]:
# Calculate accuracy for cancer predictions
cancer_predictions = predicted_df.filter(pl.col("final_label") == "CANCER")
if len(cancer_predictions) > 0:
    correct = cancer_predictions.filter(pl.col("is_cancer") == True).height
    total = cancer_predictions.height
    accuracy = correct / total * 100
    print(f"\nCancer prediction accuracy: {accuracy:.1f}% ({correct}/{total})")

# Calculate accuracy for non-cancer predictions
non_cancer_predictions = predicted_df.filter(pl.col("final_label") == "NON_CANCER")
if len(non_cancer_predictions) > 0:
    correct_non = non_cancer_predictions.filter(pl.col("is_cancer") == False).height
    total_non = non_cancer_predictions.height
    accuracy_non = correct_non / total_non * 100
    print(f"Non-cancer prediction accuracy: {accuracy_non:.1f}% ({correct_non}/{total_non})")



Cancer prediction accuracy: 73.9% (153/207)
Non-cancer prediction accuracy: 34.2% (66/193)


In [18]:
# Show false positives (predicted cancer but is_cancer=False)
false_positives = predicted_df.filter(
    (pl.col("final_label") == "CANCER") & (pl.col("is_cancer") == False)
)
print(f"\n=== False Positives ({len(false_positives)}) ===")
if len(false_positives) > 0:
    print(false_positives.select(["title", "tissue", "disease", "source_name", "final_label", "is_cancer"]))



=== False Positives (2) ===
shape: (2, 6)
┌───────────────────────────┬────────┬──────────────────┬────────────────┬─────────────┬───────────┐
│ title                     ┆ tissue ┆ disease          ┆ source_name    ┆ final_label ┆ is_cancer │
│ ---                       ┆ ---    ┆ ---              ┆ ---            ┆ ---         ┆ ---       │
│ str                       ┆ str    ┆ str              ┆ str            ┆ str         ┆ i64       │
╞═══════════════════════════╪════════╪══════════════════╪════════════════╪═════════════╪═══════════╡
│ GSM6176564: Adjacent      ┆ null   ┆ Non-tumor Tissue ┆ Mammary tissue ┆ CANCER      ┆ 0         │
│ Non-Tumor…                ┆        ┆                  ┆                ┆             ┆           │
│ RNA-Seq of canine mammary ┆ tumor  ┆ null             ┆ null           ┆ CANCER      ┆ 0         │
│ tumo…                     ┆        ┆                  ┆                ┆             ┆           │
└───────────────────────────┴────────┴──────────

In [19]:
# Show false negatives (predicted non-cancer but is_cancer=True)
false_negatives = predicted_df.filter(
    (pl.col("final_label") == "NON_CANCER") & (pl.col("is_cancer") == True)
)
print(f"\n=== False Negatives ({len(false_negatives)}) ===")
if len(false_negatives) > 0:
    print(false_negatives.select(["title", "tissue", "disease", "source_name", "final_label", "is_cancer"]))



=== False Negatives (79) ===
shape: (79, 6)
┌───────────────────┬──────────────────┬──────────────┬──────────────────┬─────────────┬───────────┐
│ title             ┆ tissue           ┆ disease      ┆ source_name      ┆ final_label ┆ is_cancer │
│ ---               ┆ ---              ┆ ---          ┆ ---              ┆ ---         ┆ ---       │
│ str               ┆ str              ┆ str          ┆ str              ┆ str         ┆ i64       │
╞═══════════════════╪══════════════════╪══════════════╪══════════════════╪═════════════╪═══════════╡
│ RNA-Seq of canine ┆ normal           ┆ null         ┆ null             ┆ NON_CANCER  ┆ 1         │
│ mammary matc…     ┆                  ┆              ┆                  ┆             ┆           │
│ 1519_L1_litter179 ┆ cutaneous        ┆ null         ┆ null             ┆ NON_CANCER  ┆ 1         │
│                   ┆ melanoma         ┆              ┆                  ┆             ┆           │
│ NextSeq 500       ┆ null             ┆ null 

In [20]:
# Define columns to export (include med_reason)
export_cols = [
    "title", 
    "tissue", 
    "disease", 
    "source_name", 
    "cancer_type",
    "final_label", 
    "is_cancer",
    "med_label",
    "med_reason",
    "confidence_category",
    "run_accession",
    "bioproject"
]

# Filter to only columns that exist in the dataframe
export_cols = [c for c in export_cols if c in predicted_df.columns]


In [21]:
# Export False Positives to Excel
# (predicted CANCER but actually is_cancer=False)
false_positives = predicted_df.filter(
    (pl.col("final_label") == "CANCER") & (pl.col("is_cancer") == False)
)

if len(false_positives) > 0:
    false_positives.select(export_cols).write_excel("outputs/false_positives.xlsx")
    print(f"✓ Exported {len(false_positives)} false positives to outputs/false_positives.xlsx")
else:
    print("No false positives found")


✓ Exported 2 false positives to outputs/false_positives.xlsx


In [22]:
# Export False Negatives to Excel
# (predicted NON_CANCER but actually is_cancer=True)
false_negatives = predicted_df.filter(
    (pl.col("final_label") == "NON_CANCER") & (pl.col("is_cancer") == True)
)

if len(false_negatives) > 0:
    false_negatives.select(export_cols).write_excel("outputs/false_negatives.xlsx")
    print(f"✓ Exported {len(false_negatives)} false negatives to outputs/false_negatives.xlsx")
else:
    print("No false negatives found")


✓ Exported 79 false negatives to outputs/false_negatives.xlsx


In [23]:
all_non_cancer = predicted_df.filter(
    (pl.col("is_cancer") == False)
)

all_cancer_uncertain = predicted_df.filter(
    (pl.col('final_label') == "UNCERTAIN")
)

# all_non_cancer
all_cancer_uncertain

title,single-cell/bulk,tissue,disease,experiment_accession,bioproject,biosample,sample_accession,study_accession,library_construction_protocol,source_name,run_accession,is_cancer,source_name_norm,tissue_norm,disease_norm,cancer_in_sample_name,negative_in_sample_name,onco_trap_in_sample_name,cancer_in_source_name,negative_in_source_name,cancer_in_tissue,negative_in_tissue,cancer_in_disease,negative_in_disease,n_cancer_mentions,n_negative_mentions,regex_label,regex_reason,med_label,med_reason,med_source_columns,confidence_category,final_label
str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,u32,u32,str,str,str,str,str,str,str


In [24]:
    # Calculate all confusion matrix components
true_positives = predicted_df.filter(
    (pl.col("final_label") == "CANCER") & (pl.col("is_cancer") == True)
)
true_negatives = predicted_df.filter(
    (pl.col("final_label") == "NON_CANCER") & (pl.col("is_cancer") == False)
)
false_positives = predicted_df.filter(
    (pl.col("final_label") == "CANCER") & (pl.col("is_cancer") == False)
)
false_negatives = predicted_df.filter(
    (pl.col("final_label") == "NON_CANCER") & (pl.col("is_cancer") == True)
)

tp, tn, fp, fn = len(true_positives), len(true_negatives), len(false_positives), len(false_negatives)

print(f"\n=== Export Summary ===")
print(f"True Positives:  {tp}")
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")

print(f"\n=== Percentages ===")
# TP% = TP / (TP + FN) aka Recall/Sensitivity
print(f"True Positives:  {tp / (tp + fn) * 100:.1f}% (of actual positives)")
# TN% = TN / (TN + FP) aka Specificity
print(f"True Negatives:  {tn / (tn + fp) * 100:.1f}% (of actual negatives)")
# FP% = FP / (FP + TN)
print(f"False Positives: {fp / (fp + tn) * 100:.1f}% (of actual negatives)")
# FN% = FN / (FN + TP)
print(f"False Negatives: {fn / (fn + tp) * 100:.1f}% (of actual positives)")

print(f"\nColumns exported: {export_cols}")



=== Export Summary ===
True Positives:  153
True Negatives:  66
False Positives: 2
False Negatives: 79

=== Percentages ===
True Positives:  65.9% (of actual positives)
True Negatives:  97.1% (of actual negatives)
False Positives: 2.9% (of actual negatives)
False Negatives: 34.1% (of actual positives)

Columns exported: ['title', 'tissue', 'disease', 'source_name', 'final_label', 'is_cancer', 'med_label', 'med_reason', 'confidence_category', 'run_accession', 'bioproject']


In [25]:
# Check what columns we actually have after classification
print("Available columns:", predicted_df.columns)

# Check if med_reason exists and what it contains
if "med_reason" in predicted_df.columns:
    print("\nSample med_reason values:")
    print(predicted_df.select(["med_label", "med_reason"]).head(10))
else:
    print("\n⚠️ med_reason column does NOT exist!")

Available columns: ['title', 'single-cell/bulk', 'tissue', 'disease', 'experiment_accession', 'bioproject', 'biosample', 'sample_accession', 'study_accession', 'library_construction_protocol', 'source_name', 'run_accession', 'is_cancer', 'source_name_norm', 'tissue_norm', 'disease_norm', 'cancer_in_sample_name', 'negative_in_sample_name', 'onco_trap_in_sample_name', 'cancer_in_source_name', 'negative_in_source_name', 'cancer_in_tissue', 'negative_in_tissue', 'cancer_in_disease', 'negative_in_disease', 'n_cancer_mentions', 'n_negative_mentions', 'regex_label', 'regex_reason', 'med_label', 'med_reason', 'med_source_columns', 'confidence_category', 'final_label']

Sample med_reason values:
shape: (10, 2)
┌────────────┬─────────────────────────────────┐
│ med_label  ┆ med_reason                      │
│ ---        ┆ ---                             │
│ str        ┆ str                             │
╞════════════╪═════════════════════════════════╡
│ CANCER     ┆ cancer_terms: canine cortisol

In [ ]:
from pathlib import Path
DATA_PATH = Path("data/combined_metadata_noncancer_removed.csv")

cancer_rules, non_cancer_rules = get_default_target_rules()
existing_rules = cancer_rules + non_cancer_rules

nlp = initialize_medspacy_pipeline(cancer_rules, non_cancer_rules)

if DATA_PATH.exists():
    all_samples = pl.read_csv(
        str(DATA_PATH),
        schema_overrides={"group": pl.Utf8},
        infer_schema_length=0,
    )
    unique_diseases = all_samples.select("disease").unique().to_series().to_list()

    auto_rules, skipped = generate_disease_rules(unique_diseases, nlp, existing_rules)

In [ ]:
for rule in auto_rules:
    print(rule)

## Debugging the Classifier

Right now I'm thinking that the main issue is with tokenization, so we're splitting multiword patterns up in unexpected ways. Still need to investigate further

In [29]:
predicted_df.filter((pl.col('final_label') == 'NON_CANCER') & (pl.col('is_cancer') == True))

title,single-cell/bulk,tissue,disease,experiment_accession,bioproject,biosample,sample_accession,study_accession,library_construction_protocol,source_name,run_accession,is_cancer,source_name_norm,tissue_norm,disease_norm,cancer_in_sample_name,negative_in_sample_name,onco_trap_in_sample_name,cancer_in_source_name,negative_in_source_name,cancer_in_tissue,negative_in_tissue,cancer_in_disease,negative_in_disease,n_cancer_mentions,n_negative_mentions,regex_label,regex_reason,med_label,med_reason,med_source_columns,confidence_category,final_label
str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,u32,u32,str,str,str,str,str,str,str
"""RNA-Seq of canine mammary matc…","""bulk""","""normal""",null,"""SRX4634736""","""PRJNA489087""",null,"""SRS3743633""","""SRP159466""",null,null,"""SRR7779525""",1,"""""","""normal""","""""",false,true,false,false,false,false,true,false,false,0,1,"""likely_non_cancer""","""neg-context""","""NON_CANCER""","""non_cancer_terms: normal | fou…","""tissue""","""confirmed_non_cancer""","""NON_CANCER"""
"""1519_L1_litter179""","""bulk""","""cutaneous melanoma""",null,"""SRX21119593""","""PRJNA997124""",null,"""SRS18386154""","""SRP450700""",null,null,"""SRR25381351""",1,"""""","""cutaneous melanoma""","""""",false,false,false,false,false,true,false,false,false,1,0,"""uncertain_weak_signal""","""weak-cancer-signal""","""NO_SIGNAL""","""no_relevant_terms""","""""","""uncertain_weak_signal""","""NON_CANCER"""
"""NextSeq 500 paired end sequenc…","""bulk""",null,null,"""DRX274419""","""PRJDB11471""","""SAMD00294424""","""DRS256590""","""DRP009015""",null,null,"""DRR284962""",1,"""""","""""","""""",false,false,false,false,false,false,false,false,false,0,0,"""uncertain_no_signal""","""""","""NO_SIGNAL""","""no_relevant_terms""","""""","""uncertain_no_signal""","""NON_CANCER"""
"""RNA-seq of canine DLBCL""","""bulk""","""DLBCL""",null,"""SRX3896467""","""PRJNA448988""","""SAMN08874665""","""SRS3133150""","""SRP137798""",null,null,"""SRR6953259""",1,"""""","""dlbcl""","""""",false,false,false,false,false,false,false,false,false,0,0,"""uncertain_no_signal""","""""","""NO_SIGNAL""","""no_relevant_terms""","""""","""uncertain_no_signal""","""NON_CANCER"""
"""Canine bladder tumor RNAseq""","""bulk""","""bladder mucosa""",null,"""SRX1532712""","""PRJNA308949""","""SAMN04417766""","""SRS1249824""","""SRP068516""",null,null,"""SRR3104207""",1,"""""","""bladder mucosa""","""""",true,false,false,false,false,false,false,false,false,0,0,"""likely_cancer""","""""","""NO_SIGNAL""","""no_relevant_terms""","""""","""likely_cancer""","""NON_CANCER"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Illumina NovaSeq 6000 paired e…","""bulk""","""neoplastic lymph node""","""canine multicentric high-grade…","""DRX438674""","""PRJDB15540""","""SAMD00588953""","""DRS286209""","""DRP009757""",null,null,"""DRR453591""",1,"""""","""neoplastic lymph node""","""canine multicentric high-grade…",false,false,false,false,false,false,false,true,false,1,0,"""uncertain_weak_signal""","""weak-cancer-signal""","""NO_SIGNAL""","""no_relevant_terms""","""""","""uncertain_weak_signal""","""NON_CANCER"""
"""Transcriptome of Bos indicus:b…","""bulk""","""Horn epithelial""",null,"""SRX4947450""","""PRJNA497667""","""SAMN10273289""","""SRS3989745""","""SRP167079""",null,null,"""SRR8121162""",1,"""""","""horn epithelial""","""""",true,false,false,false,false,false,false,false,false,0,0,"""likely_cancer""","""""","""NO_SIGNAL""","""no_relevant_terms""","""""","""likely_cancer""","""NON_CANCER"""
"""Illumina HiSeq 4000 paired end…","""bulk""",null,null,"""ERX2157135""","""PRJEB22239""","""SAMEA104221593""","""ERS1880611""","""ERP024586""","""Omental cells, from female Wis…",null,"""ERR2099830""",1,"""""","""""","""""",false,false,false,false,false,false,false,false,false,0,0,"""uncertain_no_signal""","""""","""NO_SIGNAL""","""no_relevant_terms""","""""","""uncertain_no_signal""","""NO

In [30]:
# Get false negatives
false_negatives = predicted_df.filter(
    (pl.col('final_label') == 'NON_CANCER') & (pl.col('is_cancer') == True)
)

# Look at a specific example with "cutaneous melanoma"
melanoma_cases = false_negatives.filter(pl.col('tissue').str.contains('melanoma'))

# Show key columns to understand what's happening
melanoma_cases.select([
    'tissue',
    'tissue_norm',  # Check if normalized column exists
    'disease',
    'disease_norm',
    'source_name',
    'source_name_norm',
    'med_label',
    'med_reason',
    'med_source_columns'  # This should show which columns had matches
]).head(5)

tissue,tissue_norm,disease,disease_norm,source_name,source_name_norm,med_label,med_reason,med_source_columns
str,str,str,str,str,str,str,str,str
"""cutaneous melanoma""","""cutaneous melanoma""",null,"""""",null,"""""","""NO_SIGNAL""","""no_relevant_terms""",""""""


In [33]:
# Find rows where tissue actually contains melanoma
melanoma_rows = false_negatives.filter(pl.col('tissue').str.contains('melanoma', literal=False))

if len(melanoma_rows) > 0:
    example = melanoma_rows[0].to_dicts()[0]

    print(f"tissue (original): {example.get('tissue', 'N/A')}")
    print(f"tissue_norm: {example.get('tissue_norm', 'COLUMN DOES NOT EXIST')}")
    print(f"disease (original): {example.get('disease', 'N/A')}")
    print(f"disease_norm: {example.get('disease_norm', 'COLUMN DOES NOT EXIST')}")
    print(f"\nmed_reason: {example.get('med_reason', 'N/A')}")
    print(f"med_label: {example.get('med_label', 'N/A')}")

    # Most importantly - show what columns exist
    print(f"\nAvailable columns: {melanoma_rows.columns}")
else:
    print("No rows found with 'melanoma' in tissue column")
    print("\nShowing all unique tissue values in false negatives:")
    print(false_negatives.select('tissue').unique())

tissue (original): cutaneous melanoma
tissue_norm: cutaneous melanoma
disease (original): None
disease_norm: 

med_reason: no_relevant_terms
med_label: NO_SIGNAL

Available columns: ['title', 'single-cell/bulk', 'tissue', 'disease', 'experiment_accession', 'bioproject', 'biosample', 'sample_accession', 'study_accession', 'library_construction_protocol', 'source_name', 'run_accession', 'is_cancer', 'source_name_norm', 'tissue_norm', 'disease_norm', 'cancer_in_sample_name', 'negative_in_sample_name', 'onco_trap_in_sample_name', 'cancer_in_source_name', 'negative_in_source_name', 'cancer_in_tissue', 'negative_in_tissue', 'cancer_in_disease', 'negative_in_disease', 'n_cancer_mentions', 'n_negative_mentions', 'regex_label', 'regex_reason', 'med_label', 'med_reason', 'med_source_columns', 'confidence_category', 'final_label']


In [34]:
from functions import get_nlp

nlp = get_nlp()

# Test the exact text
test_text = "cutaneous melanoma"
doc = nlp(test_text)

print(f"Text: '{test_text}'")
print(f"Entities found: {[(ent.text, ent.label_) for ent in doc.ents]}")

# Test just "melanoma"
doc2 = nlp("melanoma")
print(f"\nText: 'melanoma'")
print(f"Entities found: {[(ent.text, ent.label_) for ent in doc2.ents]}")

2026-03-04 12:23:14.964 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=991] [doc 0] Token 0 'cutaneous' marked as sentence start (span begin)
2026-03-04 12:23:14.965 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=991] Token/tag mapping: [(cutaneous, True), (melanoma, False)]
2026-03-04 12:23:14.967 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=992] [doc 0] Token 0 'melanoma' marked as sentence start (span begin)
2026-03-04 12:23:14.967 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=992] Token/tag mapping: [(melanoma, True)]


Text: 'cutaneous melanoma'
Entities found: [('melanoma', 'CANCER')]

Text: 'melanoma'
Entities found: [('melanoma', 'CANCER')]
